# Build Coupled Wflow-SFINCS Model

Build the Austin inland coupled base models with explicit HydroMT-Wflow and HydroMT-SFINCS steps.


> **Stage Contract**
>
> Requires: `01_region_setup.ipynb` domain artifacts, the selected Wflow HUC watershed artifact, reviewed USGS observation/calibration gages, static DEM/LULC/soil rasters, and the Wflow HydroMT data catalog inputs.
>
> Produces: the selected encompassing-HUC Wflow submodel, the selected SMART-DS SFINCS coverage box, boundary source points where Wflow-native rivers enter that box, and Wflow gauge outputs at the SFINCS source points.
>
> Next: `05_create_scenarios.ipynb` stages event forcing against these base models.


In [ ]:
from os.path import join
import os
from pathlib import Path

os.environ.pop("DEBUG", None)

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import yaml
from hydromt_sfincs import DATADIR, SfincsModel

from design_events.config import load_runtime
from sfincs_runs.build_base import (
    add_inland_outflow_boundary,
    plan_inland_sfincs_base,
    plan_inland_sfincs_domain_set,
    write_inland_sfincs_domain_set_manifest,
    write_inland_sfincs_handoff_locations,
)
from wflow_runs import (
    build_wflow_data_catalog,
    build_wflow_submodel,
    plot_wflow_basemap,
    plot_wflow_ldd_components,
    wflow_catalog_source_readiness,
    write_wflow_domain_set_manifest,
)
from wflow_runs.notebook import wflow_domain_set_summary, wflow_subbasin_review_table


def configread(path):
    with open(path, encoding="utf-8") as handle:
        return yaml.safe_load(handle)


In [ ]:
# Load local packages and this Location Workspace.
notebook_root = Path.cwd().resolve()
location_root = notebook_root.parents[1]
repo_root = location_root.parents[1]
location_name = location_root.name
config, paths = load_runtime(location_root / "config.yaml")
# Wflow domain review policy is workflow policy, not source/model-recipe YAML.
wflow_domain_review_required = True
config["wflow"]["domain_set"]["review_required"] = wflow_domain_review_required


def location_path(value):
    path = Path(value)
    return path if path.is_absolute() else location_root / path


build_wflow = True
build_all_wflow_submodels = True
write_wflow_catalog = True
write_domain_set_manifest = True
run_sfincs_domain_build = True
force_sfincs_domain_build = False
force_wflow_river_build = False
rebuild_wflow_with_boundary_handoffs = True
force_wflow_boundary_gauge_build = False

sfincs_data_catalog = location_root / "data/static/data_catalogue.yaml"
wflow_data_catalog = location_path(config["wflow"]["data_catalog"])
wflow_base_root = location_path(config["wflow"]["base_model_root"])
wflow_network_path = Path(paths["usgs_streamgage_network_geojson"])

data_cols = ["category", "crs", "data_type", "uri"]


## Part 1 — Coupled Domain Plans

Wflow and SFINCS use different geometry meanings: Wflow is hydrologic and can extend upstream; SFINCS is hydraulic coverage around each SMART-DS component.


### Step 1 · Wflow watershed and SFINCS coverage plans

Plan Wflow from the configured hydrologic boundary (`encompassing_huc`) and plan SFINCS from the selected SMART-DS coverage box. The Wflow HUC domain supplies the upstream routing area; the SFINCS domain IDs written here are the hydraulic coupling targets used later by Wflow gauges.


In [ ]:
wflow_build_plan, wflow_domain_plan, domain_summary = wflow_domain_set_summary(config, location_root)
sfincs_base_plan = plan_inland_sfincs_base(config, paths)
sfincs_domain_plan = plan_inland_sfincs_domain_set(config, paths)

if sfincs_domain_plan.status == "ready":
    sfincs_domain_manifest = write_inland_sfincs_domain_set_manifest(sfincs_domain_plan, config, paths)
else:
    sfincs_domain_manifest = None

write_domain_set_manifest = wflow_domain_plan.status == "ready"
if write_domain_set_manifest:
    wflow_domain_manifest = write_wflow_domain_set_manifest(wflow_domain_plan, config, paths)
else:
    wflow_domain_manifest = None

domain_set = yaml.safe_load(location_path(config["sfincs_domain_set"]["domain_manifest"]).read_text()) if sfincs_domain_manifest else {"domains": []}
sfincs_domains = list(domain_set["domains"])

display(domain_summary)
display(wflow_subbasin_review_table(wflow_domain_plan))
display(pd.Series({
    "wflow_domain_status": wflow_domain_plan.status,
    "wflow_domain_manifest": str(wflow_domain_manifest.relative_to(repo_root)) if wflow_domain_manifest else "review_required",
    "sfincs_domain_status": sfincs_domain_plan.status,
    "sfincs_domain_manifest": str(sfincs_domain_manifest.relative_to(repo_root)) if sfincs_domain_manifest else "review_required",
    "sfincs_domain_count": sfincs_domain_plan.domain_count,
    "sfincs_handoff_count": sfincs_domain_plan.handoff_count,
}, name="domain_set_plans"))


### Step 2 · Hydrologic boundary and handoff map

Plot the selected Wflow watershed boundary, selected SMART-DS coverage, and planned stream/coverage-box handoff points before any model build. This is the hydrologic handoff check: Wflow owns the upstream watershed, while SFINCS owns only the hydraulic coverage box.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

sfincs_domain_layers = [gpd.read_file(location_path(domain["region"])) for domain in sfincs_domains]
sfincs_domain_gdf = pd.concat(sfincs_domain_layers, ignore_index=True)
sfincs_domain_gdf = gpd.GeoDataFrame(sfincs_domain_gdf, geometry="geometry", crs=sfincs_domain_layers[0].crs).to_crs(4326)
sfincs_label = "SFINCS coverage box" if len(sfincs_domain_gdf) == 1 else "SFINCS coverage boxes"
sfincs_coverage_geom = sfincs_domain_gdf.geometry.union_all()

# Wflow build geometry is the HUC/HUC-union watershed selected in 01_region_setup,
# not the older NHDPlus review fabric that only documents candidate catchments.
wflow_watershed_layers = []
for submodel in wflow_domain_plan.submodels:
    watershed_path = submodel.get("subbasin_geometry")
    if watershed_path:
        watershed = gpd.read_file(location_path(watershed_path)).to_crs(4326)
        watershed["wflow_submodel_id"] = submodel["wflow_submodel_id"]
        wflow_watershed_layers.append(watershed)

if not wflow_watershed_layers:
    watershed_path = config["static_sources"]["wflow_collection_extent"]["watersheds"]
    wflow_watershed_layers.append(gpd.read_file(location_path(watershed_path)).to_crs(4326))

wflow_watershed_gdf = gpd.GeoDataFrame(
    pd.concat(wflow_watershed_layers, ignore_index=True),
    geometry="geometry",
    crs=wflow_watershed_layers[0].crs,
).to_crs(4326)
wflow_watershed_gdf.boundary.plot(ax=ax, color="#2b6cb0", linewidth=1.4, label="Wflow HUC watershed boundary")

study = gpd.read_file(location_root / "data/static/aoi/evaluation_footprint.geojson").to_crs(4326)
selected_source_subregion_ids = {
    str(domain.get("exposure_subregion_id")).strip()
    for domain in sfincs_domains
    if str(domain.get("exposure_subregion_id", "")).strip()
}
if selected_source_subregion_ids and "subregion_id" in study.columns:
    selected_study = study[study["subregion_id"].astype(str).isin(selected_source_subregion_ids)].copy()
else:
    # Fallback for footprints without source subregion ids: keep only pieces that touch the selected SFINCS coverage.
    candidate_study_components = study.explode(index_parts=False).reset_index(drop=True)
    selected_study = candidate_study_components[candidate_study_components.intersects(sfincs_coverage_geom)].copy()
if selected_study.empty:
    raise RuntimeError("No SMART-DS footprint geometry matched the selected SFINCS domains")
selected_study_geom = selected_study.geometry.union_all()
selected_study.boundary.plot(ax=ax, color="black", linewidth=1.0, label="selected SMART-DS footprint")

sfincs_domain_gdf.boundary.plot(ax=ax, color="red", linewidth=1.4, linestyle="--", label=sfincs_label)

handoff_records = []
for submodel in wflow_domain_plan.submodels:
    for point in submodel.get("handoff_points", []):
        handoff_records.append({
            "sfincs_handoff_id": point["sfincs_handoff_id"],
            "wflow_submodel_id": submodel["wflow_submodel_id"],
            "sfincs_domain_id": point.get("sfincs_domain_id"),
            "uparea_km2": point.get("uparea_km2"),
            "geometry": gpd.points_from_xy([point["lon"]], [point["lat"]])[0],
        })

handoff_plan_gdf = gpd.GeoDataFrame(handoff_records, geometry="geometry", crs="EPSG:4326")
if not handoff_plan_gdf.empty:
    handoff_plan_gdf.plot(
        ax=ax,
        marker="D",
        color="crimson",
        markersize=35,
        label="planned/pre-snap SFINCS/Wflow boundary handoff",
    )

wflow_watershed_geom = wflow_watershed_gdf.geometry.union_all()
planned_handoff_boundary_distance_m = None
if not handoff_plan_gdf.empty:
    model_crs = config["sfincs"].get("model_crs", "EPSG:32614")
    boundary = sfincs_domain_gdf.to_crs(model_crs).geometry.union_all().boundary
    distances = handoff_plan_gdf.to_crs(model_crs).geometry.distance(boundary)
    planned_handoff_boundary_distance_m = float(distances.max())

display(pd.Series({
    "wflow_watershed_features": int(len(wflow_watershed_gdf)),
    "selected_smart_ds_subregions": ", ".join(sorted(selected_source_subregion_ids)) if selected_source_subregion_ids else "spatial_intersection_fallback",
    "sfincs_coverage_covers_selected_footprint": bool(sfincs_coverage_geom.covers(selected_study_geom)),
    "wflow_watershed_covers_sfincs_coverage": bool(wflow_watershed_geom.covers(sfincs_coverage_geom)),
    "planned_boundary_handoff_count": int(len(handoff_plan_gdf)),
    "max_planned_handoff_distance_from_sfincs_boundary_m": round(planned_handoff_boundary_distance_m, 3) if planned_handoff_boundary_distance_m is not None else "no planned handoffs",
}, name="domain_plan_hydrologic_boundary_check"))

ax.set_title(f"Coupled domain plan: Wflow watershed feeding selected {sfincs_label.lower()}")
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.legend(loc="best")


## Part 2 — HydroMT-Wflow Native Build

Wflow is built first because its DEM/LDD-derived `staticgeoms/rivers.geojson` is the authoritative stream linework for SFINCS source placement.


### Step 3 · HydroMT-Wflow data catalog readiness

HydroMT-Wflow needs the local DEM-derived hydrography basemap, landcover, soil parameter maps, and event forcing sources before the submodels can be built.


In [ ]:
wflow_catalog_path = build_wflow_data_catalog(config, paths) if write_wflow_catalog else wflow_data_catalog
wflow_source_readiness = pd.DataFrame(wflow_catalog_source_readiness(wflow_catalog_path))
missing_required_wflow_source = (
    wflow_source_readiness["required_for_build"].fillna(False).astype(bool)
    & wflow_source_readiness["local_file"].fillna(False).astype(bool)
    & ~wflow_source_readiness["exists"].fillna(False).astype(bool)
)
required_wflow_sources_missing = wflow_source_readiness[missing_required_wflow_source]

if not required_wflow_sources_missing.empty:
    missing_lines = [
        f"{row.source}: {row.uri}"
        for row in required_wflow_sources_missing.itertuples(index=False)
    ]
    display(wflow_source_readiness)
    raise FileNotFoundError(
        "Missing required HydroMT-Wflow source files before build:\n"
        + "\n".join(missing_lines)
        + "\nRun locations/austin/02_flood/01_region_setup.ipynb through the Terrain, Landcover, and Wflow static inputs cell first. "
        + "For a fresh Wflow static pull, launch that notebook with FLOOD_RM_FETCH_DEM=1; landcover fetch is enabled by default."
    )

wflow_source_readiness


### Step 4 · Wflow build steps

Inspect the HydroMT-Wflow workflow. `build_wflow_submodel` keeps this syntax but replaces the template region with each reviewed `subbasin + uparea` outlet region.


In [ ]:
wf_config = configread(str(location_path(config["wflow"]["build_config"])))
print(wf_config.keys())
pd.DataFrame({"step": [next(iter(step)) for step in wf_config.get("steps", [])]})


### Step 5 · Build or reuse Wflow submodels

Each Wflow submodel is the configured hydrologic domain from `wflow.domain_set` (for `encompassing_huc`, one WBD HUC/HUC-union basin per SFINCS coverage box). This first pass guarantees that HydroMT-Wflow-native rivers, basins, gauges, and static maps are available before SFINCS source placement.


In [ ]:
if wflow_domain_plan.status != "ready":
    raise RuntimeError(f"Wflow Domain Set requires review before build: {wflow_domain_plan.status}: {wflow_domain_plan.issues}")
if sfincs_domain_plan.status != "ready":
    raise RuntimeError(f"SFINCS coverage domains require review before build: {sfincs_domain_plan.status}: {sfincs_domain_plan.issues}")

selected_submodels = wflow_domain_plan.submodels if build_all_wflow_submodels else wflow_domain_plan.submodels[:1]
wflow_native_build_summary = []
if build_wflow:
    for submodel in selected_submodels:
        submodel_id = submodel["wflow_submodel_id"]
        river_fn = wflow_base_root / submodel_id / "staticgeoms/rivers.geojson"
        wflow_native_build_summary.append(
            build_wflow_submodel(
                config,
                paths,
                submodel_id=submodel_id,
                force=force_wflow_river_build or not river_fn.exists(),
                write_catalog=False,
            )
        )

wflow_models = {summary["wflow_submodel_id"]: summary["model"] for summary in wflow_native_build_summary}
pd.DataFrame([{k: v for k, v in summary.items() if k != "model"} for summary in wflow_native_build_summary])


### Step 6 · Wflow static map inventory

List the HydroMT-Wflow static maps and static geometries produced by `setup_basemaps`, `setup_rivers`, and `setup_gauges` before trusting any handoff point.


In [ ]:
wflow_inventory_rows = []
wflow_native_handoff_rows = []

for submodel in selected_submodels:
    submodel_id = submodel["wflow_submodel_id"]
    wf = wflow_models[submodel_id]

    for name in wf.staticmaps.data.data_vars:
        wflow_inventory_rows.append({"wflow_submodel_id": submodel_id, "artifact": "staticmaps", "name": name})
    for name, geom in wf.geoms.data.items():
        wflow_inventory_rows.append({"wflow_submodel_id": submodel_id, "artifact": "staticgeoms", "name": name, "feature_count": len(geom)})

    planned_handoffs = submodel.get("sfincs_handoff_ids", [])
    native_handoffs = wf.geoms.data.get("gauges_sfincs")
    native_handoff_ids = [] if native_handoffs is None else sorted(native_handoffs["name"].astype(str).tolist())
    wflow_native_handoff_rows.append({
        "wflow_submodel_id": submodel_id,
        "planned_boundary_crossings": len(planned_handoffs),
        "native_snapped_wflow_gauges": len(native_handoff_ids),
        "native_gauge_ids": ", ".join(native_handoff_ids),
    })

# Planned crossings mark every SFINCS boundary/river intersection; HydroMT snaps only those represented on routed Wflow cells.
display(pd.DataFrame(wflow_native_handoff_rows))
pd.DataFrame(wflow_inventory_rows)


### Step 7 · Wflow native basemap plots

Plot elevation, river cells, HydroMT-Wflow river vectors, basins, gauges, and SFINCS coverage boxes using the built Wflow model objects and their native `staticmaps` / `geoms` artifacts.


In [ ]:
for submodel_id, wf in wflow_models.items():
    fig, ax = plot_wflow_basemap(wf, sfincs_domains=sfincs_domain_gdf, figsize=(8, 6), title=f"{submodel_id} Wflow native base map")
    fig.savefig(wflow_base_root / submodel_id / "wflow_native_basemap.png", dpi=450, bbox_inches="tight")


### Step 8 · Wflow LDD component plots

Inspect the flow-aware layers that control handoff placement: elevation, upstream area, stream order, and local drain direction.


In [ ]:
for submodel_id, wf in wflow_models.items():
    fig, _ = plot_wflow_ldd_components(wf)
    fig.savefig(wflow_base_root / submodel_id / "wflow_ldd_components_native.png", dpi=300, bbox_inches="tight")


## Part 3 — HydroMT-SFINCS Coverage Build

SFINCS domains are hydraulic coverage boxes around SMART-DS components. They are not assumed to be full watersheds.


### Step 9 · Initialize SFINCS coverage models

Each SMART-DS coverage box gets its own `SfincsModel` root. The discharge sources are added after Wflow-native rivers are available.


In [ ]:
sfincs_models = {}
sfincs_model_rows = []

for domain in sfincs_domains:
    sfincs_domain_id = domain["sfincs_domain_id"]
    root = location_path(domain["base_model_root"])
    sfincs_model_ready = (root / "sfincs.inp").exists()
    sfincs_model_mode = "w+" if force_sfincs_domain_build or not sfincs_model_ready else "r+"
    if sfincs_model_mode == "w+" and not run_sfincs_domain_build:
        raise FileNotFoundError(
            f"SFINCS base model is missing for {sfincs_domain_id}: {root}. "
            "Set run_sfincs_domain_build=True before opening a new coverage model."
        )
    sf = SfincsModel(
        root=str(root),
        mode=sfincs_model_mode,
        data_libs=[str(sfincs_data_catalog)],
    )
    sfincs_models[sfincs_domain_id] = sf
    sfincs_model_rows.append({
        "sfincs_domain_id": sfincs_domain_id,
        "region": str(location_path(domain["region"]).relative_to(repo_root)),
        "base_model_root": str(root.relative_to(repo_root)),
        "model_ready": sfincs_model_ready,
        "model_mode": sfincs_model_mode,
        "handoff_source_count": len(domain["handoff_source_ids"]),
    })

sf_data = ["dem_region", "landcover_region", "hydrologic_soil_group", "saturated_conductivity"]
display(next(iter(sfincs_models.values())).data_catalog._to_dataframe().loc[sf_data, data_cols])
pd.DataFrame(sfincs_model_rows)


### Step 10 · SFINCS build configuration

Use the standard HydroMT-SFINCS build recipe, but omit automatic river inflow/outflow detection because discharge points are supplied from Wflow-native river/domain intersections.


In [ ]:
sf_config = configread(str(location_root / config["includes"]["sfincs_build"]))
sf_config["setup_dep"]["datasets_dep"] = [{"elevtn": "dem_region"}]
sf_config["setup_subgrid"]["datasets_dep"] = [{"elevtn": "dem_region"}]
sf_config["setup_subgrid"]["datasets_rgh"] = [{"lulc": "landcover_region"}]
sf_config.pop("setup_river_inflow", None)
sf_config.pop("setup_river_outflow", None)
print(sf_config.keys())


### Step 11 · Build or reuse SFINCS grids, masks, subgrid, and Wflow source points

The source GeoJSON is written after the SFINCS grid exists so the discharge points can be stored in the model, but the coordinates come from Wflow-native river intersections with the coverage-box boundary.


In [ ]:
sfincs_build_rows = []
handoff_layers = []

for domain in sfincs_domains:
    domain_id = domain["sfincs_domain_id"]
    sf = sfincs_models[domain_id]
    root = location_path(domain["base_model_root"])
    region = {"geom": str(location_path(domain["region"]))}
    sf_config_domain = yaml.safe_load(yaml.safe_dump(sf_config))
    sf_config_domain["setup_grid_from_region"]["region"] = region

    sfincs_model_ready = (root / "sfincs.inp").exists()
    build_this_sfincs_domain = run_sfincs_domain_build and (force_sfincs_domain_build or not sfincs_model_ready)
    if build_this_sfincs_domain:
        sf.build(steps=[
            {"grid.create_from_region": sf_config_domain["setup_grid_from_region"]},
            {"elevation.create": {"elevation_list": [{"elevation": "dem_region"}]}},
            {"mask.create_active": sf_config_domain["setup_mask_active"]},
            {"subgrid.create": {
                "elevation_list": [{"elevation": "dem_region"}],
                "roughness_list": [{"lulc": "landcover_region", "reclass_table": str(Path(DATADIR) / "lulc/esa_worldcover_mapping.csv")}],
                "write_dep_tif": True,
                "nr_subgrid_pixels": config["sfincs"].get("nr_subgrid_pixels", 6),
            }},
        ])
        add_inland_outflow_boundary(sf)

    handoff_locations = root / "gis/wflow_handoff_sources.geojson"
    write_inland_sfincs_handoff_locations(
        config,
        paths,
        output=handoff_locations,
        handoff_source_ids=domain["handoff_source_ids"],
        domain_region=location_path(domain["region"]),
        sfincs_domain_id=domain_id,
    )
    src = gpd.read_file(handoff_locations)
    sf.discharge_points.set_locations(src)
    sf.write()
    handoff_layers.append(src)
    sfincs_build_rows.append({
        "sfincs_domain_id": domain_id,
        "built": build_this_sfincs_domain,
        "root": str(sf.root.path),
        "handoff_source_count": len(domain["handoff_source_ids"]),
        "handoff_status": "written",
    })

pd.DataFrame(sfincs_build_rows)


### Step 12 · SFINCS coverage QA plots

Plot each SFINCS coverage grid with reviewed gages, HydroMT-Wflow-native river vectors, and the Wflow-derived boundary source points.


In [ ]:
handoff_gdf = pd.concat(handoff_layers, ignore_index=True)
handoff_gdf = gpd.GeoDataFrame(handoff_gdf, geometry="geometry", crs=handoff_layers[0].crs)

for domain in sfincs_domains:
    domain_id = domain["sfincs_domain_id"]
    sf = sfincs_models[domain_id]
    sf.elevation.data["dep"].attrs.update(long_name="elevation", units="m")
    fig, ax = sf.plot_basemap(figsize=(8, 6), plot_bounds=True, plot_geoms=True)
    if wflow_network_path.exists():
        gpd.read_file(wflow_network_path).to_crs(sf.crs).plot(
            ax=ax,
            marker="o",
            facecolor="none",
            edgecolor="black",
            markersize=25,
            label="reviewed USGS gage",
        )
    river_layers = []
    for submodel_id in domain["wflow_submodel_ids"]:
        river_fn = wflow_base_root / submodel_id / "staticgeoms/rivers.geojson"
        if river_fn.exists():
            river_layers.append(gpd.read_file(river_fn).to_crs(sf.crs))
    if river_layers:
        rivers = gpd.GeoDataFrame(pd.concat(river_layers, ignore_index=True), geometry="geometry", crs=river_layers[0].crs)
        rivers.plot(ax=ax, color="royalblue", linewidth=1.1, alpha=0.8, label="Wflow-native rivers")
    src_fn = location_path(domain["base_model_root"]) / "gis/wflow_handoff_sources.geojson"
    src = gpd.read_file(src_fn).to_crs(sf.crs)
    src.plot(ax=ax, marker="D", color="crimson", markersize=45, label="Wflow boundary source")
    ax.legend(loc="best")
    fig.savefig(join(sf.root.path, "sfincs_basemap.png"), dpi=450, bbox_inches="tight")


### Step 13 · SFINCS handoff source table

Review the source coordinates and their river-intersection provenance before rebuilding Wflow gauges.


In [ ]:
handoff_columns = [
    "sfincs_domain_id",
    "sfincs_handoff_id",
    "wflow_submodel_id",
    "site_no",
    "handoff_placement",
    "handoff_location_review_status",
    "stream_boundary_river_source",
    "stream_boundary_candidate_count",
]
handoff_columns = [column for column in handoff_columns if column in handoff_gdf.columns]
handoff_gdf[handoff_columns]


## Part 4 — Coupling Back Into Wflow

After SFINCS writes boundary source points, Wflow is rebuilt with gauge outputs at exactly those source points. The Wflow model may be much larger than the SFINCS boxes.


### Step 14 · Rebuild Wflow gauges at SFINCS boundary sources

This second Wflow pass keeps the same HydroMT-Wflow subbasins and static maps, but the `setup_gauges` SFINCS layer now reads the SFINCS source GeoJSONs.


In [ ]:
wflow_build_summary = []
if build_wflow:
    for submodel in selected_submodels:
        wflow_build_summary.append(
            build_wflow_submodel(
                config,
                paths,
                submodel_id=submodel["wflow_submodel_id"],
                force=rebuild_wflow_with_boundary_handoffs and force_wflow_boundary_gauge_build,
                write_catalog=False,
            )
        )

wflow_models = {summary["wflow_submodel_id"]: summary["model"] for summary in wflow_build_summary}
pd.DataFrame([{k: v for k, v in summary.items() if k != "model"} for summary in wflow_build_summary])


### Step 15 · Final coupled Wflow basemaps

Plot the native Wflow model with SFINCS coverage boxes and SFINCS source points overlaid. This is the main check that the upstream Wflow model feeds the right hydraulic coverage box boundaries.


In [ ]:
for submodel_id, wf in wflow_models.items():
    fig, ax = plot_wflow_basemap(wf, sfincs_domains=sfincs_domain_gdf, figsize=(8, 6), title=f"{submodel_id} coupled Wflow base map")
    handoff_gdf.to_crs(wf.crs).plot(ax=ax, marker="x", color="crimson", markersize=65, label="SFINCS source")
    ax.legend(loc="best")
    fig.savefig(wflow_base_root / submodel_id / "wflow_basemap.png", dpi=450, bbox_inches="tight")


### Step 16 · Final Wflow LDD and gauge QA

Replot the LDD components after gauge insertion and list the Wflow gauge layers that will produce discharge for SFINCS.


In [ ]:
gauge_rows = []
handoff_contract_rows = []
model_crs = config["sfincs"].get("model_crs", config["project"].get("model_crs", "EPSG:32614"))
max_allowed_handoff_snap_m = float(config["sfincs"].get("grid_resolution_m", 100))

for submodel_id, wf in wflow_models.items():
    fig, _ = plot_wflow_ldd_components(wf)
    fig.savefig(wflow_base_root / submodel_id / "wflow_ldd_components.png", dpi=300, bbox_inches="tight")
    for name, geom in wf.geoms.data.items():
        if name.startswith("gauges") or name.startswith("subcatchment"):
            gauge_rows.append({"wflow_submodel_id": submodel_id, "geometry_layer": name, "feature_count": len(geom)})

    sfincs_gauges = wf.geoms.data.get("gauges_sfincs")
    if sfincs_gauges is None or sfincs_gauges.empty:
        raise ValueError(f"{submodel_id} has no gauges_sfincs layer for SFINCS discharge handoff.")

    sources = handoff_gdf[handoff_gdf["wflow_submodel_id"].astype(str).eq(str(submodel_id))].to_crs(model_crs)
    if sources.empty:
        raise ValueError(f"{submodel_id} has no SFINCS handoff sources to couple back to Wflow.")

    gauges = sfincs_gauges.to_crs(model_crs).copy()
    for _, source in sources.iterrows():
        distances = gauges.geometry.distance(source.geometry)
        nearest_index = distances.idxmin()
        nearest = gauges.loc[nearest_index]
        handoff_contract_rows.append({
            "wflow_submodel_id": submodel_id,
            "sfincs_handoff_id": source["sfincs_handoff_id"],
            "wflow_gauge_index": nearest.get("index"),
            "wflow_gauge_handoff_id": nearest.get("sfincs_handoff_id", nearest.get("name")),
            "source_to_wflow_gauge_m": round(float(distances.loc[nearest_index]), 3),
        })

handoff_contract = pd.DataFrame(handoff_contract_rows)
handoff_contract["source_count_per_wflow_gauge"] = handoff_contract.groupby(
    ["wflow_submodel_id", "wflow_gauge_index"]
)["sfincs_handoff_id"].transform("count")
bad_snap = handoff_contract[handoff_contract["source_to_wflow_gauge_m"].gt(max_allowed_handoff_snap_m)]
if not bad_snap.empty:
    raise ValueError(
        "SFINCS handoff sources are farther than one SFINCS grid cell from their nearest Wflow gauges: "
        + ", ".join(bad_snap["sfincs_handoff_id"].astype(str))
    )

display(handoff_contract)
pd.DataFrame(gauge_rows)
